In [2]:
# Run this cell beforehand so you do not see any warnings
import warnings
warnings.filterwarnings('ignore')

In [1]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn import preprocessing
import faiss
import numpy as np
import pickle

c:\Users\Admin\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset from a JSON file into a Pandas DataFrame  
data = pd.read_json('./data.json.txt')  

# Remove unnecessary columns
df = data.drop(columns=["author", "link", 'tag'])  

# Print the total number of unique Machine Learning papers  
print("Number of Machine Learning papers: ", df.id.unique().shape[0])  

# Display the first few rows of the modified DataFrame  
df.head() 

Number of Machine Learning papers:  41000


,day,id,month,summary,title,year
0,1,1802.00209v1,2,We propose an architecture for VQA which utili...,Dual Recurrent Attention Units for Visual Ques...,2018
1,12,1603.03827v1,3,Recent approaches based on artificial neural n...,Sequential Short-Text Classification with Recu...,2016
2,2,1606.00776v2,6,We introduce the multiresolution recurrent neu...,Multiresolution Recurrent Neural Networks: An ...,2016
3,23,1705.08142v2,5,Multi-task learning is motivated by the observ...,Learning what to share between loosely related...,2017
4,7,1709.02349v2,9,We present MILABOT: a deep reinforcement learn...,A Deep Reinforcement Learning Chatbot,2017


In [3]:
# Load the pre-trained SentenceTransformer model
model = SentenceTransformer('distilbert-base-nli-stsb-mean-tokens')

# Check if a GPU (CUDA) is available and move the model to GPU if possible
if torch.cuda.is_available():
    model = model.to(torch.device("cuda"))

# Print the device the model is running on (CPU or GPU)
print(model.device)

c:\Users\Admin\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--distilbert-base-nli-stsb-mean-tokens. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3148.05it/s]


cpu


In [4]:
embeddings = model.encode(df.summary.to_list()[:2000], show_progress_bar=True)
with open('./new_embeddings.pickle', 'wb') as pkl:
  pickle.dump(embeddings, pkl)

Batches: 100%|██████████| 63/63 [03:45<00:00,  3.58s/it]


In [5]:
embeddings = model.encode(df.summary.to_list()[:2000], show_progress_bar=True)
with open('./new_embeddings.pickle', 'wb') as pkl:
  pickle.dump(embeddings, pkl)

Batches: 100%|██████████| 63/63 [03:49<00:00,  3.64s/it]


In [ ]:

with open('./new_embeddings.pickle', 'rb') as pkl:
  embeddings = pickle.load(pkl)


length = len(embeddings)
print ('length: ', length)
print('Shape of the one embedding: ', embeddings[0].shape)
print("Number of embeddings:", len(embeddings))
print("Dimension of each embedding:", embeddings[0].shape)
print("Total values stored:", len(embeddings) * embeddings[0].shape[0])
## Task 5: Data Preparation and Helper Methods
le = preprocessing.LabelEncoder()
df['id'] = le.fit_transform(df['id'])
df.head(2)

def id2info(df, I, column):
    return [list(df[df.id == idx][column]) for idx in I]

## Task 6: Set up the Index
embeddings = np.array(embeddings).astype("float32")
print (embeddings.shape[1])
index = faiss.IndexFlatL2(embeddings.shape[1])
index = faiss.IndexIDMap(index)
index.add_with_ids(embeddings, df['id'][:length])

print("Number of embeddings in the Faiss index: ", index.ntotal)
## Task 7: Search with a Summary
df.iloc[1337, [3, 1]]

D, I = index.search(np.array([embeddings[1337]]), k=10)
pd.DataFrame({'L2 distance': D.flatten().tolist(), 'ML paper IDs': I.flatten().tolist(), 'ML paper titles': id2info(df, I.flatten(), 'title'), 'Summaries': id2info(df, I.flatten(), 'summary')}).head(10)

## Task 8: Search with a Prompt

user_query = "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data"
embed = model.encode(list(user_query))
D, I = index.search(np.array([embed]).squeeze().astype("float32"), k=10)

results = {'L2 distances':D.flatten().tolist(), 'ML paper IDs':I.flatten().tolist(), "Titles": id2info(df, I.flatten(), 'title'), "Summaries": id2info(df, I.flatten(), 'summary')}

pd.DataFrame(results).head(10)
# End

length:  2000
Shape of the one embedding:  (768,)
Number of embeddings: 2000
Dimension of each embedding: (768,)
Total values stored: 1536000
768
Number of embeddings in the Faiss index:  2000


,L2 distances,ML paper IDs,Titles,Summaries
0,398.834991,26890,[Abstract Syntax Networks for Code Generation ...,[Tasks like code generation and semantic parsi...
1,401.845032,30018,[Dual Rectified Linear Units (DReLUs): A Repla...,"[In this paper, we introduce a novel type of R..."
2,403.243469,7184,[KSU KDD: Word Sense Induction by Clustering i...,[We describe our language-independent unsuperv...
3,403.410797,28659,[Topic supervised non-negative matrix factoriz...,[Topic models have been extensively used to or...
4,404.912415,34786,[Don't Just Assume; Look and Answer: Overcomin...,[A number of studies have found that today's V...
5,406.426086,30468,[Regularizing and Optimizing LSTM Language Mod...,"[Recurrent neural networks (RNNs), such as lon..."
6,410.384155,16734,[Learning the Dimensionality of Word Embeddings],[We describe a method for learning word embedd...
7,410.741302,12050,[HD-CNN: Hierarchical Deep Convolutional Neura...,"[In image classification, visual separability ..."
8,414.264191,11912,[Taking into Account the Differences between A...,[Actively sampled data can have very different...
9,414.365814,31758,[Self-Guiding Multimodal LSTM - when we do not...,"[In this paper, a self-guiding multimodal LSTM..."
